# 9. Save a query under a name

`picsure::saveQueryByName(session, query, name, overwrite = FALSE)` submits a query server-side and associates a human-readable name with it. The function returns the UUID of the saved query, which you can later round-trip through `loadQueryByID()` / `runQueryByID()` (notebooks 5 and 6).

Saved queries are scoped to the authenticated user, so this only works against an authorized platform (`Platform$BDC_AUTHORIZED`, `Platform$BDC_DEV_AUTHORIZED`). On open platforms the call errors.

The name accepts letters, digits, spaces, and `- _ \ / ? + = [ ] . ( ) : " '`, up to 255 chars. Passing `overwrite = TRUE` updates an existing named record instead of erroring on the duplicate.

In [ ]:
library(picsure)

In [ ]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [ ]:
auth_hpds_session <- picsure::connect(
  platform = picsure::Platform$BDC_AUTHORIZED,
  token    = my_token
)

## Build a query to save

Same OR-of-ANDs shape from notebooks 4 and 8: FHS participants aged 30-40, split by sex and re-unioned.

In [ ]:
facets <- picsure::facets(auth_hpds_session)
picsure::addFacet(facets, "dataset_id", c("phs000810", "phs000007"))

In [ ]:
results <- picsure::searchDictionary(auth_hpds_session, "age", facets = facets)

age5_phs000007 <- results[results$display == "age5" & results$name == "phv00177938", ]
age5_phs000007_clause <- picsure::buildClause(
  age5_phs000007$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

In [ ]:
fhs_facet <- picsure::facets(auth_hpds_session)
picsure::addFacet(fhs_facet, "dataset_id", "phs000007")

fhs_sex_results <- picsure::searchDictionary(auth_hpds_session, "phv00253990", facets = fhs_facet)
fhs_sex_results

In [ ]:
fhs_sex_male_clause <- picsure::buildClause(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Male")
)
fhs_male_and_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_sex_male_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

fhs_sex_female_clause <- picsure::buildClause(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Female")
)
fhs_female_and_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_sex_female_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

fhs_female_or_male_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_female_and_30_to_40, fhs_male_and_30_to_40),
  operator = picsure::GroupOperator$OR
)

fhs_female_or_male_30_to_40$to_query_json()

In [ ]:
picsure::runQuery(auth_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_female_or_male_30_to_40))
# Verified with UI: 77 +-3

## Save it

`saveQueryByName()` returns the new query's UUID. `overwrite = TRUE` makes the call idempotent if you re-run the notebook.

In [ ]:
# Persist a Query that both filters and selects output columns;
# includeConcepts is carried through save / load.
query_to_save <- picsure::buildQuery(
  phenotypicFilter = fhs_female_or_male_30_to_40,
  includeConcepts  = c(age5_phs000007$conceptPath[[1]], fhs_sex_results$conceptPath[[1]])
)

saved_id <- picsure::saveQueryByName(
  auth_hpds_session,
  query_to_save,
  "Created from API adapters - fhs_FemaleAges30to40_OR_MaleAges30to40",
  overwrite = TRUE
)
saved_id

## Round-trip: load it back and re-run

Proves the save took -- the loaded query handle should produce the same count as the in-memory original.

In [ ]:
reloaded <- picsure::loadQueryByID(auth_hpds_session, saved_id)
picsure::runQuery(auth_hpds_session, reloaded)